# Phase 4: In-Silico Perturbation & Causal Inference

## 📝 Executive Overview
This notebook acts as our "Digital Petri Dish." Having fine-tuned the Geneformer foundation model to predict dose-dependent transcriptomic responses, we now utilize the frozen model to simulate a targeted genetic knockout.

By programmatically resolving a target Ensembl ID to its latent token representation, we can scan our baseline single-cell sequences for native expression. We then apply an **In-Silico Perturbation**—masking the target token with a `<pad>` tensor—forcing the bidirectional attention mechanism to recalculate the cell's transcriptomic state without the presence of the target driver.

### 🎯 Key Objectives
1. **Dynamic Vocabulary Resolution:** Automatically fetch the 104M Geneformer token dictionary to map target HGNC symbols to their integer embedding IDs.
2. **Architecture Rehydration:** Rebuild the custom unrolled TransformerEngine wrapper and load the fine-tuned `.pt` weights into VRAM.
3. **Deterministic Inference:** Lock the network into evaluation mode (`torch.no_grad()`) and execute the digital knockout to calculate the causal Delta ($\Delta$) shift.

In [ ]:
# =====================================================================
# ⚙️ GLOBAL PIPELINE BOUNDARY CONFIGURATION
# =====================================================================
import os
import sys
import pickle
import urllib.request

# 1. Target Configuration (Fully Generalizable)
TARGET_GENE_SYMBOL = "ABCG2"
TARGET_ENSEMBL_ID  = "ENSG00000118777"
PAD_TOKEN_ID       = 0
MAX_SEQ_LEN        = 2048

# 2. IO Pathing
INPUT_DATA_PATH     = "data/geneformer_tuning_input.pkl"
MODEL_WEIGHTS_PATH  = "data/gout_geneformer_finetuned.pt"
DICT_PATH           = "data/token_dictionary_gc104M.pkl"

BIONEMO_RECIPE_PATH = "/content/bionemo-framework/bionemo-recipes/models/geneformer/src/geneformer"

# 3. Dynamic Token Resolution
print(f"🔍 Resolving Token ID for target: {TARGET_GENE_SYMBOL} ({TARGET_ENSEMBL_ID})...")

if not os.path.exists(DICT_PATH):
    print("   📥 Downloading official Geneformer token dictionary...")
    urllib.request.urlretrieve("https://huggingface.co/ctheodoris/Geneformer/resolve/main/geneformer/token_dictionary_gc104M.pkl", DICT_PATH)

with open(DICT_PATH, "rb") as f:
    gene_token_dict = pickle.load(f)

TARGET_TOKEN_ID = gene_token_dict.get(TARGET_ENSEMBL_ID)

if TARGET_TOKEN_ID is None:
    raise ValueError(f"❌ CRITICAL: {TARGET_ENSEMBL_ID} not found in the pre-trained dictionary.")

print(f"✅ Target precisely mapped to embedding Token ID: {TARGET_TOKEN_ID}")

## 🔌 Architecture Reconstruction & Weight Rehydration
Because cloud computing environments are stateless, we must instantiate the exact unrolled model blueprint utilized during training before we can map our saved state dictionary into the GPU VRAM.

In [ ]:
import torch
import torch.nn as nn

# Link to NVIDIA native recipes
if BIONEMO_RECIPE_PATH not in sys.path:
    sys.path.append(BIONEMO_RECIPE_PATH)

from modeling_bert_te import BertModel, TEBertConfig

# 1. Initialize strict hardware configuration map
model_config = TEBertConfig(
    hidden_size=768, num_attention_heads=12, num_hidden_layers=6,
    vocab_size=40000, max_position_embeddings=2048
)
model_config.position_embedding_type = "absolute"
model_config.is_decoder = False

# 2. Rebuild the Unrolled Architecture
class GeneformerForRegression(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.bert = BertModel(config)
        self.regression_head = nn.Linear(config.hidden_size, 1)

    def forward(self, input_ids):
        hidden_states = self.bert.embeddings(input_ids=input_ids)
        batch_size, seq_length = input_ids.shape
        attention_mask = torch.zeros((batch_size, 1, 1, seq_length), dtype=hidden_states.dtype, device=hidden_states.device)

        for layer_module in self.bert.encoder.layer:
            layer_outputs = layer_module(hidden_states, attention_mask)
            hidden_states = layer_outputs[0] if isinstance(layer_outputs, tuple) else layer_outputs

        cell_embedding = hidden_states[:, 0, :]
        return self.regression_head(cell_embedding)

# 3. Hardware Routing & Rehydration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
regression_model = GeneformerForRegression(model_config)

print(f"📂 Loading pre-trained weights from: {MODEL_WEIGHTS_PATH}")
regression_model.load_state_dict(torch.load(MODEL_WEIGHTS_PATH, map_location=device))
regression_model.to(device)

# 4. Lock parameters for deterministic predictions
regression_model.eval()
print("🧠 Rehydration Complete: Model is locked in Inference Mode.")

## 🔬 Executing the In-Silico Knockout
The script below isolates a single cell demonstrating native expression of our target gene. It establishes a baseline state prediction, artificially masks the target token to sever its bidirectional attention pathways, and recalculates the continuous dosage state.

In [ ]:
# 1. Load the Single-Cell Input Arrays
with open(INPUT_DATA_PATH, 'rb') as f:
    data = pickle.load(f)

print(f"🔍 Scanning {len(data['tokens'])} parsed cells for native {TARGET_GENE_SYMBOL} expression...")

# 2. Target Isolation Search
target_cell_idx = -1
for idx, sequence in enumerate(data['tokens']):
    if TARGET_TOKEN_ID in sequence:
        target_cell_idx = idx
        break

if target_cell_idx == -1:
    print(f"\n❌ Experiment Failed: {TARGET_GENE_SYMBOL} is not natively expressed in the current pilot subset.")
    print("    -> Recommendation: Increase MAX_SUBSET_CELLS in Phase 1 to capture a wider biological distribution.")
else:
    print(f"✅ Found native {TARGET_GENE_SYMBOL} representation at Cell Index: {target_cell_idx}")

    # Ensure sequence fits within memory bounds
    baseline_sequence = data['tokens'][target_cell_idx][:MAX_SEQ_LEN]

    # 3. Apply the Digital Knockout
    perturbed_sequence = [PAD_TOKEN_ID if token == TARGET_TOKEN_ID else token for token in baseline_sequence]

    # Package for hardware execution
    baseline_tensor = torch.tensor([baseline_sequence], dtype=torch.long).to(device)
    perturbed_tensor = torch.tensor([perturbed_sequence], dtype=torch.long).to(device)

    # 4. Deterministic Inference Execution
    with torch.no_grad():
        baseline_prediction = regression_model(baseline_tensor).item()
        perturbed_prediction = regression_model(perturbed_tensor).item()

    # Calculate absolute transcriptomic shift
    delta_shift = perturbed_prediction - baseline_prediction

    print("\n📊 --- IN-SILICO PERTURBATION RESULTS ---")
    print(f"  Target Gene Masked:     {TARGET_GENE_SYMBOL} (ID: {TARGET_TOKEN_ID})")
    print(f"  Baseline Cell State:    {baseline_prediction:.4f}")
    print(f"  Perturbed Cell State:   {perturbed_prediction:.4f}")
    print(f"  Δ (Delta) Causal Shift: {delta_shift:.4f}")